# Chunking: passages, quotes and summaries

How a chapter becomes the records that are embedded and searched.

See the [README](README.md) for the why and the how.

In [ ]:
# Run from anywhere inside the repository: find its root so `marginalia` and the test helpers import,
# and read .env (the same settings the services use).
import os, sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / 'pyproject.toml').exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print(ROOT)

In [ ]:
from tests.pdfs import make_book, chapter
from marginalia.ingest.book import load_book
book = load_book(make_book([chapter('CHAPTER I', 'the river', n=14), chapter('CHAPTER II', 'the axe', n=14)],
                           title='Chunks'))
text = book.chapters[0].text
len(text), text[:200]

## `chunk_text`: about 1,000 characters of whole sentences, overlapping

In [ ]:
from marginalia.ingest.chunking import chunk_text
spans = chunk_text(text, target=1000, overlap=150)
for s in spans:
    print(f'[{s.start:5}-{s.end:5}] {len(s.text):4} chars | {s.text[:60]!r} ... {s.text[-40:]!r}')
print('overlap of the first two:', spans[0].end - spans[1].start, 'chars')

## Smaller or larger chunks

In [ ]:
for target in (300, 1000, 2000):
    sp = chunk_text(text, target=target, overlap=target // 7)
    print(f'target {target:5}: {len(sp):3} chunks, sizes {min(len(s.text) for s in sp)}-{max(len(s.text) for s in sp)}')

## Quotes
Lines of dialogue are extracted as their own records, so "who said ..." questions can search speech directly.

In [ ]:
from marginalia.ingest.quotes import extract_quotes
dialogue = ('He stopped at the door. “I am a titular counsellor and my daughter walks the streets for us, ” said '
            'Marmeladov, looking at the floor for a long while before he went on. ') * 3
for q in extract_quotes(dialogue, 'double_curly', min_chars=20)[:3]:
    print(q)

## The records that go to the index
One record per chunk, quote and chapter summary, each with its book, index version, part, chapter and position. Summaries lead with the chapter label and title.

In [ ]:
from marginalia.ingest.pipeline import build_records, summary_records
records = build_records(book.chapters, 'demo-book', 1, book.quote_style)
records += summary_records(book.chapters, ['Summary of chapter one.', 'Summary of chapter two.'], 'demo-book', 1)
from collections import Counter
print(Counter(r.kind for r in records))
r = records[0]
print(r.payload())
print(next(r for r in records if r.kind == 'summary').text)